# 4주차 — GPT구조와_프롬프트_비교

- 100분 제출형 실습 · 가설 → 구현/실험 → 증거 → 해석
- 학생용 TODO 완성 후 다음 셀 실행 · TODO에서 중단되는 것은 의도된 동작임
- 환경·모델 로딩·반복 학습은 지원 코드로 제공
- 답안은 개조식으로 기록 · 결과 방향보다 비교 조건과 해석을 평가
- Colab: 런타임 → 런타임 유형 변경 → GPU 권장
- 4주차 GPT-3 원형 API 이외의 기본 실습은 API 키 불필요


In [1]:
%pip -q install "transformers==4.57.6" "peft==0.17.1" "accelerate==1.10.1" "numpy>=1.26,<2.3" "pandas>=2.2,<3" "matplotlib>=3.8,<3.11" "scikit-learn>=1.4,<1.8"
%pip -q install "ftfy>=6.2,<7" requests


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 31.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.3 MB/s eta 0:00:00


In [2]:
import os, sys, json, math, time, random, hashlib, gc, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

SEED = 42
set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(4, os.cpu_count() or 1))
RESULT_DIR = Path("lab_results_week4")
RESULT_DIR.mkdir(exist_ok=True)
ENV = {"python":sys.version.split()[0], "torch":torch.__version__,
       "device":str(DEVICE), "gpu":torch.cuda.get_device_name(0) if DEVICE.type=="cuda" else None,
       "seed":SEED, "smoke_test":SMOKE}
display(pd.DataFrame([ENV]))

def sync():
    if DEVICE.type == "cuda": torch.cuda.synchronize()

def cleanup():
    gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

def save_json(name, obj):
    (RESULT_DIR / name).write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")

def download_results(label):
    import shutil
    archive = shutil.make_archive(label, "zip", RESULT_DIR)
    try:
        from google.colab import files
        files.download(archive)
    except ImportError:
        print("저장 위치:", Path(archive).resolve())

STUDENT_ID=""  # 학번 입력
STUDENT_NAME=""  # 이름 입력
ANSWERS={}
PASSED=set()


KeyboardInterrupt: 

## 문제 4-1. GPT-1 방식 Post-LN에서 warmup이 정말 필요한가?

- 질문: GPT-1의 Post-LN과 GPT-2의 Pre-LN을 코드와 통제 실험으로 구분할 수 있는가?
- 구현: 아래 작은 블록의 두 forward 경로 완성
- 실험: 폭·깊이·공유 가중치·입력·정답 고정, LN 위치만 변경, 3개 seed 비교
- 제출: 구조 식별 근거, 완성 코드, 층별 gradient 그래프, 단정할 수 없는 점
- 식: Pre-LN $x+F(LN(x))$, Post-LN $LN(x+F(x))$
- 초기 gradient 실험은 GPT-1/2 전체 사전학습 재현이 아님
- 추가 실험: Post-LN/Pre-LN × warmup 없음/있음 × 최대 학습률 2개 × seed 3개
- GPT-1 방식의 작은 무작위 초기화 causal LM 사용 · 학습 완료된 GPT-1의 추가 학습과 구분

In [ ]:
import inspect, copy
from transformers.models.openai.modeling_openai import Block as GPT1Block
from transformers.models.gpt2.modeling_gpt2 import GPT2Block
for name,cls in [("GPT-1",GPT1Block),("GPT-2",GPT2Block)]:
    source=inspect.getsource(cls.forward)
    print("\n",name,"forward:\n",source)
    (RESULT_DIR/f"{name}_forward.txt").write_text(source,encoding="utf-8")


In [ ]:
class NormBlock(nn.Module):
    def __init__(self,pre=True,d=32):
        super().__init__(); self.pre=pre
        self.ln1=nn.LayerNorm(d); self.ln2=nn.LayerNorm(d)
        self.attn=nn.MultiheadAttention(d,4,dropout=0.,batch_first=True)
        self.ffn=nn.Sequential(nn.Linear(d,4*d),nn.GELU(),nn.Linear(4*d,d))
    def attention(self,x):
        mask=torch.triu(torch.ones(x.shape[1],x.shape[1],dtype=torch.bool,device=x.device),1)
        return self.attn(x,x,x,attn_mask=mask,need_weights=False)[0]
    def forward(self,x):
        if self.pre:
            raise NotImplementedError("TODO: Pre-LN attention·FFN 잔차 경로 구현 필요")
        else:
            raise NotImplementedError("TODO: Post-LN attention·FFN 잔차 경로 구현 필요")


In [ ]:
for pre in [True,False]:
    block=NormBlock(pre); probe=torch.randn(2,8,32,requires_grad=True)
    result=block(probe)
    assert result.shape==probe.shape and torch.isfinite(result).all()
    result.square().mean().backward()
    assert probe.grad is not None and torch.isfinite(probe.grad).all()
DEPTH=8
gradient_rows=[]
for seed in [0,1,2]:
    torch.manual_seed(seed)
    pre_blocks=nn.ModuleList([NormBlock(True) for _ in range(DEPTH)])
    post_blocks=copy.deepcopy(pre_blocks)
    for block in post_blocks: block.pre=False
    x=torch.randn(2,8,32); target=torch.randn(2,8,32)
    final_norm=nn.LayerNorm(32); head=nn.Linear(32,32)
    for label,blocks in [("pre",pre_blocks),("post",post_blocks)]:
        head.zero_grad(); final_norm.zero_grad(); h=x
        for block in blocks: h=block(h)
        loss=F.mse_loss(head(final_norm(h)),target); loss.backward()
        for i,block in enumerate(blocks):
            gradient_rows.append({"seed":seed,"norm":label,"layer":i+1,
                                  "gradient_norm":float(block.attn.in_proj_weight.grad.norm())})
gradient_table=pd.DataFrame(gradient_rows)
for label,group in gradient_table.groupby("norm"):
    agg=group.groupby("layer").gradient_norm.agg(["mean","std"])
    plt.errorbar(agg.index,agg["mean"],yerr=agg["std"],label=label,capsize=3)
plt.xlabel("Layer"); plt.ylabel("Attention gradient norm"); plt.legend(); plt.show()
gradient_table.to_csv(RESULT_DIR/"ln_gradient.csv",index=False)
PASSED.add("4-1_shape_gradient")


### 추가 실험 — Post-LN에서 warmup의 필요성 검증

- 가설: 동일 최대 학습률에서 warmup이 초기 손실 급증·기울기 크기에 미치는 영향
- 연구 근거: [Xiong et al., 2020](https://proceedings.mlr.press/v119/xiong20b.html)의 LN 위치·초기 gradient 분석
- 범위: GPT-1 방식 Post-LN 블록을 갖는 축소 모델의 처음부터 학습 · 원형 GPT-1 재현 아님
- 통제: 동일 초기 가중치·데이터·미니배치 순서·optimizer·step 수·최대 학습률
- 추가 통제: 두 구조 모두 최종 LN 사용, dropout·gradient clipping·weight decay 미사용
- 학습률: 0.001, 0.01을 실행 전 고정 · seed 0/1/2 · 40-step · warmup 10-step
- 과제: 작은 합성 토큰 수열의 next-token prediction · 실제 자연어 corpus와의 차이 명시
- TODO: 첫 update부터 선형 증가하는 학습률 함수 완성
- 제출: learning rate·train loss·gradient 그래프, seed별 결과표, Post-LN에서 warmup 효과 해석
- 판단 기준: 비유한 값 발생 여부, 초기 10-step 최대 손실, 최대 gradient norm, 최종 validation loss
- 손실 2배 초과 지표는 관찰용 기준 · 수학적 발산과 구분
- warmup은 초기 누적 학습률도 줄임 · 결과가 좋아도 원인을 LN만으로 설명 불가
- 차이가 없으면 해당 작은 모델·학습률·40-step에서 차이를 확인하지 못한 것으로 기록


In [ ]:
def learning_rate_at(step, peak_lr, warmup_steps):
    # step: 0부터 시작 · warmup=0이면 항상 peak_lr
    # warmup=10이면 첫 update 0.1배, 열 번째 update부터 1배
    raise NotImplementedError("TODO: 선형 warmup 후 일정 학습률 구현 필요")


In [ ]:
assert learning_rate_at(0,1.,0)==1.
assert abs(learning_rate_at(0,1.,10)-0.1)<1e-8
assert learning_rate_at(9,1.,10)==learning_rate_at(20,1.,10)==1.
WARMUP_CONFIG={"steps":40,"warmup_steps":10,"peak_lrs":[0.001,0.01],
               "seeds":[0,1,2],"depth":6,"width":32,"batch":8,"length":16,"vocab":32}
save_json("warmup_config.json",WARMUP_CONFIG)
class TinyWarmupLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed=nn.Embedding(32,32); self.pos=nn.Parameter(torch.zeros(1,16,32))
        self.blocks=nn.ModuleList([NormBlock(True) for _ in range(6)])
        self.final_ln=nn.LayerNorm(32); self.head=nn.Linear(32,32)
    def forward(self,ids):
        h=self.embed(ids)+self.pos[:,:ids.shape[1]]
        for block in self.blocks: h=block(h)
        return self.head(self.final_ln(h))

generator=torch.Generator().manual_seed(2026)
def make_sequences(n):
    start=torch.randint(0,32,(n,1),generator=generator)
    stride=torch.randint(1,5,(n,1),generator=generator)
    return (start+stride*torch.arange(17))%32
train_sequences=make_sequences(256)
validation_sequences=make_sequences(64).to(DEVICE)
batch_indices=torch.randint(0,256,(40,8),generator=generator)
warmup_rows=[]; warmup_summaries=[]
for seed in WARMUP_CONFIG["seeds"]:
    torch.manual_seed(seed)
    template=TinyWarmupLM()
    for peak_lr in WARMUP_CONFIG["peak_lrs"]:
        for norm in ["post","pre"]:
            for warmup_steps in [0,10]:
                model=copy.deepcopy(template).to(DEVICE)
                for block in model.blocks: block.pre=(norm=="pre")
                optimizer=torch.optim.AdamW(model.parameters(),lr=peak_lr,weight_decay=0.)
                run=[]; status="finite"
                for step in range(40):
                    model.train(); optimizer.zero_grad(set_to_none=True)
                    batch=train_sequences[batch_indices[step]].to(DEVICE)
                    lr=learning_rate_at(step,peak_lr,warmup_steps)
                    for group in optimizer.param_groups: group["lr"]=lr
                    loss=F.cross_entropy(model(batch[:,:-1]).reshape(-1,32),batch[:,1:].reshape(-1))
                    loss.backward()
                    grad=torch.sqrt(sum(p.grad.detach().float().square().sum() for p in model.parameters() if p.grad is not None))
                    row={"seed":seed,"peak_lr":peak_lr,"norm":norm,"warmup_steps":warmup_steps,
                         "step":step+1,"lr":lr,"loss":float(loss.detach()),"gradient_norm":float(grad)}
                    run.append(row); warmup_rows.append(row)
                    if not (torch.isfinite(loss) and torch.isfinite(grad)):
                        status="nonfinite_loss_or_gradient"; break
                    optimizer.step()
                model.eval()
                with torch.no_grad():
                    val=F.cross_entropy(model(validation_sequences[:,:-1]).reshape(-1,32),validation_sequences[:,1:].reshape(-1))
                if not torch.isfinite(val): status="nonfinite_validation"
                warmup_summaries.append({"seed":seed,"peak_lr":peak_lr,"norm":norm,"warmup_steps":warmup_steps,
                    "status":status,"completed_steps":len(run),"first_loss":run[0]["loss"],
                    "early_peak_loss":max(r["loss"] for r in run[:10]),
                    "loss_spike_over_2x":any(r["loss"]>2*run[0]["loss"] for r in run),
                    "max_gradient_norm":max(r["gradient_norm"] for r in run),
                    "final_validation_loss":float(val),"lr_sum":sum(r["lr"] for r in run)})
                del model,optimizer
warmup_curves=pd.DataFrame(warmup_rows); warmup_summary=pd.DataFrame(warmup_summaries)
warmup_curves.to_csv(RESULT_DIR/"warmup_curves.csv",index=False)
warmup_summary.to_csv(RESULT_DIR/"warmup_summary.csv",index=False)
display(warmup_summary)
display(warmup_summary.groupby(["peak_lr","norm","warmup_steps"])[
    ["early_peak_loss","max_gradient_norm","final_validation_loss","lr_sum"]].agg(["mean","std"]))
fig,axes=plt.subplots(2,3,figsize=(15,7))
for row_index,peak_lr in enumerate(WARMUP_CONFIG["peak_lrs"]):
    subset=warmup_curves[warmup_curves.peak_lr==peak_lr]
    for (norm,warm),group in subset.groupby(["norm","warmup_steps"]):
        for col,metric in enumerate(["lr","loss","gradient_norm"]):
            agg=group.groupby("step")[metric].agg(["mean","std"])
            ax=axes[row_index,col]
            ax.plot(agg.index,agg["mean"],label=f"{norm}, warmup={warm}")
            ax.fill_between(agg.index,agg["mean"]-agg["std"].fillna(0),agg["mean"]+agg["std"].fillna(0),alpha=.1)
            ax.set_title(f"peak lr={peak_lr}: {metric}"); ax.set_xlabel("Update")
            if metric=="gradient_norm": ax.set_yscale("log")
            ax.legend(fontsize=7)
plt.tight_layout(); fig.savefig(RESULT_DIR/"warmup_comparison.png",dpi=130); plt.show()
PASSED.add("4-1_warmup_comparison_executed")  # 실험 실행 확인 · 안정화 성공을 의미하지 않음


### 제출 답안

- 아래 문자열에 직접 기록 · 표/그림은 위 실행 결과와 함께 제출
- 관찰과 해석을 구분하고, 예상과 다른 결과도 그대로 기록


In [ ]:
ANSWERS["4-1"] = {
    "실행 전 가설": "",
    "GPT-1·2의 LN 위치 근거": "",
    "통제 조건": "",
    "관찰과 해석": "",
    "Post-LN warmup 효과: 학습률별 수치·seed 변동": "",
    "Pre-LN과의 차이·학습률 누적합 교란": "",
    "warmup이 항상 필수인지·추가 검증 조건": ""
}
save_json("answers.json", ANSWERS)


## 문제 4-2. 어떤 출력을 정답으로 인정할 것인가?

- 과제: 감정 분류·이름 추출·쉼표 형식 변환, 각 평가 문항 4개
- 구현: zero-shot/2-shot 프롬프트 함수 완성
- 직접 수정: 평가 문항 최소 1개를 자신이 만든 문항으로 교체
- 시연 문항과 평가 문항의 입력 중복 금지
- 내용 정답 탐색과 엄격 일치 점수의 차이 확인 · 원문도 직접 판정
- 제출: 문항·정답·채점 규칙·프롬프트


In [ ]:
TASKS={
 "sentiment":{"instruction":"Classify the sentiment. Output only positive or negative.",
  "demos":[("I enjoyed every minute of the movie.","positive"),("The meal was terrible.","negative")],
  "eval":[("The service was wonderful.","positive"),("I regret buying this broken product.","negative"),
          ("The book made me happy.","positive"),("The trip was disappointing.","negative")]},
 "extraction":{"instruction":"Extract the person's name. Output only the name.",
  "demos":[("Alice bought a blue notebook.","Alice"),("Bob arrived at noon.","Bob")],
  "eval":[("Mary found a key.","Mary"),("John visited a museum.","John"),
          ("Emma wrote a letter.","Emma"),("David lost his umbrella.","David")]},
 "format":{"instruction":"Join the words with a comma and no spaces. Output only the result.",
  "demos":[("red blue","red,blue"),("cat dog","cat,dog")],
  "eval":[("sun moon","sun,moon"),("tea coffee","tea,coffee"),
          ("pen book","pen,book"),("day night","day,night")]}
}
def score_output(task,text,answer):
    # 정답 단어 포함 점수와 엄격 일치 점수를 분리 · 최종 판정은 원문과 대조
    if task=="sentiment": match=re.search(r"\b(positive|negative)\b",text.lower())
    elif task=="extraction": match=re.search(r"\b"+re.escape(answer)+r"\b",text,re.I)
    else: match=re.search(re.escape(answer),text,re.I)
    content_correct=bool(match) and (task!="sentiment" or match.group(1)==answer)
    return {"content_correct":content_correct,"strict_match":text.strip().lower()==answer.lower()}


In [ ]:
def build_prompt(task,text,shots):
    spec=TASKS[task]
    # TODO: instruction, 시연 examples, 마지막 평가 입력/Output: 연결
    raise NotImplementedError("TODO: 프롬프트 구성 필요")


In [ ]:
for task,spec in TASKS.items():
    assert len(spec["eval"])==4 and len(spec["demos"])>=2
    assert not ({x for x,y in spec["eval"]}&{x for x,y in spec["demos"]})
    for shots in [0,2]:
        prompt=build_prompt(task,spec["eval"][0][0],shots)
        assert isinstance(prompt,str) and spec["eval"][0][0] in prompt
        assert prompt.count("Input:")==shots+1
        print(task,shots,"shot\n",prompt,"\n")
save_json("tasks.json",TASKS); PASSED.add("4-2_prompt")


### 제출 답안

- 아래 문자열에 직접 기록 · 표/그림은 위 실행 결과와 함께 제출
- 관찰과 해석을 구분하고, 예상과 다른 결과도 그대로 기록


In [ ]:
ANSWERS["4-2"] = {
    "직접 수정한 평가 문항": "",
    "내용 정답과 형식 준수 기준": "",
    "시연·평가 분리 이유": "",
    "zero-shot·2-shot 차이": ""
}
save_json("answers.json", ANSWERS)


## 문제 4-3. GPT-1·2는 같은 과제를 얼마나 수행하는가?


- 실행: GPT-1·GPT-2 × 3과제 × 4문항 × 2프롬프트 = 48회
- 통제: 같은 입력·예시·greedy 생성·최대 24개 새 토큰
- tokenizer가 다르므로 같은 토큰 수가 같은 문자 길이를 뜻하지 않음
- 영어 base 모델에 chat template 사용하지 않음
- 성공/실패 원문 확인, 오류 유형 직접 분류
- 개선 프롬프트로 동일 문항 3개 재평가
- 제출: 결과표, 오류 사례, 개선 전후 출력, 일반화의 한계


In [ ]:
MODELS=[("GPT-1","openai-community/openai-gpt","1e0d4f3028acbffb47fe933cea64619c5ec1a002"),
        ("GPT-2","openai-community/gpt2","607a30d783dfa663caf39e06633721c8d4cfcd7e")]
MAX_NEW=24
local_rows=[]
for label,model_id,revision in MODELS:
    tok=AutoTokenizer.from_pretrained(model_id,revision=revision)
    net=AutoModelForCausalLM.from_pretrained(model_id,revision=revision,use_safetensors=True).to(DEVICE).eval()
    pad_id=tok.eos_token_id if tok.eos_token_id is not None else tok.unk_token_id
    for task,spec in TASKS.items():
        for index,(text,answer) in enumerate(spec["eval"]):
            for shots in [0,2]:
                prompt=build_prompt(task,text,shots)
                inp=tok(prompt,return_tensors="pt").to(DEVICE)
                assert inp.input_ids.shape[1]+MAX_NEW<=net.config.n_positions
                with torch.no_grad():
                    generated=net.generate(**inp,max_new_tokens=MAX_NEW,do_sample=False,pad_token_id=pad_id)
                raw=tok.decode(generated[0,inp.input_ids.shape[1]:],skip_special_tokens=True)
                local_rows.append({"model":label,"model_id":model_id,"revision":revision,"task":task,
                  "item":index,"shots":shots,"prompt":prompt,"expected":answer,"raw":raw,
                  "input_tokens":inp.input_ids.shape[1],**score_output(task,raw,answer)})
    del net; cleanup()
local_table=pd.DataFrame(local_rows)
display(local_table.groupby(["model","task","shots"])[["content_correct","strict_match"]].mean())
display(local_table[["model","task","shots","expected","raw"]].head(12))
local_table.to_csv(RESULT_DIR/"gpt1_gpt2_predictions.csv",index=False)
PASSED.add("4-3_local_run")


In [ ]:
def improved_prompt(task,text):
    # TODO: 한 요인만 바꾼 개선 프롬프트 작성
    raise NotImplementedError("TODO: 개선 프롬프트 구현 필요")


In [ ]:
tok=AutoTokenizer.from_pretrained(MODELS[1][1],revision=MODELS[1][2])
net=AutoModelForCausalLM.from_pretrained(MODELS[1][1],revision=MODELS[1][2],use_safetensors=True).to(DEVICE).eval()
improved=[]
for task,spec in TASKS.items():
    text,answer=spec["eval"][0]; prompt=improved_prompt(task,text)
    inp=tok(prompt,return_tensors="pt").to(DEVICE)
    assert inp.input_ids.shape[1]+MAX_NEW<=net.config.n_positions
    with torch.no_grad(): generated=net.generate(**inp,max_new_tokens=MAX_NEW,do_sample=False,pad_token_id=tok.eos_token_id)
    raw=tok.decode(generated[0,inp.input_ids.shape[1]:],skip_special_tokens=True)
    improved.append({"task":task,"prompt":prompt,"raw":raw,"expected":answer,**score_output(task,raw,answer)})
display(pd.DataFrame(improved)); save_json("prompt_improvement.json",improved)
del net; cleanup()


### 제출 답안

- 아래 문자열에 직접 기록 · 표/그림은 위 실행 결과와 함께 제출
- 관찰과 해석을 구분하고, 예상과 다른 결과도 그대로 기록


In [ ]:
ANSWERS["4-3"] = {
    "모델·프롬프트별 관찰": "",
    "성공 또는 부분 성공 사례": "",
    "실패 유형과 원문": "",
    "개선 프롬프트의 변경 요인과 결과": "",
    "Pre-LN 효과와 혼동하면 안 되는 이유": ""
}
save_json("answers.json", ANSWERS)


## 문제 4-4. GPT-3 원형에서 같은 프롬프트는 어떻게 작동하는가?

- 원형 GPT-3 직접 비교 전용 연결부 · 최신 모델 자동 대체 없음
- 강사가 실제 접근 경로·모델 식별 근거를 지정한 뒤 실행
- 공식 기존 GPT-3 API는 종료된 상태임. 별도로 확보한 원형 모델의 호환 접근 경로가 필요함
- 미연결 시 not_run 저장 · 성공·실패·정답률을 꾸며 넣지 않음
- 규격: HTTPS POST, model/prompt/max_tokens/temperature 입력, choices[0].text 출력
- 다른 규격이면 call_gpt3를 해당 제공처 문서에 맞게 수정 필요
- 제출: 24문항의 응답 원문·반환 모델명·점수·ICL 설명
- 미연결 시: 설계와 제한사항 제출 가능. 직접 실험 완료로 채점하지 않음

In [ ]:
# 강사 설정 · 비밀키는 Colab Secrets 또는 숨김 입력으로 전달
RUN_GPT3=False
GPT3_COMPLETIONS_URL=""
GPT3_MODEL_ID=""
GPT3_MODEL_EVIDENCE=""  # 원형 GPT-3임을 확인한 문서/계약/모델 카드 근거
GPT3_EXPECTED_RESPONSE_MODEL=""
GPT3_MAX_CALLS=24
GPT3_KEY_SECRET="GPT3_API_KEY"
def call_gpt3(prompt,api_key):
    import requests
    response=requests.post(GPT3_COMPLETIONS_URL,
      headers={"Authorization":"Bearer "+api_key,"Content-Type":"application/json"},
      json={"model":GPT3_MODEL_ID,"prompt":prompt,"max_tokens":MAX_NEW,"temperature":0,"n":1},
      timeout=60,allow_redirects=False)
    if response.status_code!=200: raise RuntimeError(f"API HTTP {response.status_code}: 설정 확인 필요")
    payload=response.json()
    actual=payload.get("model")
    if actual!=GPT3_EXPECTED_RESPONSE_MODEL: raise ValueError("반환 모델 식별자 불일치")
    return payload["choices"][0]["text"],actual
gpt3_rows=[]
if RUN_GPT3:
    from urllib.parse import urlparse
    from getpass import getpass
    assert urlparse(GPT3_COMPLETIONS_URL).scheme=="https"
    assert GPT3_MODEL_ID and GPT3_MODEL_EVIDENCE and GPT3_EXPECTED_RESPONSE_MODEL
    assert GPT3_MAX_CALLS>=24
    try:
        from google.colab import userdata
        api_key=userdata.get(GPT3_KEY_SECRET)
    except Exception:
        api_key=getpass("API key (출력/파일 저장 없음): ")
    try:
        for row in local_rows:
            if row["model"]!="GPT-2": continue
            raw,actual=call_gpt3(row["prompt"],api_key)
            gpt3_rows.append({"model":"GPT-3 original","requested_model":GPT3_MODEL_ID,"returned_model":actual,
               "task":row["task"],"item":row["item"],"shots":row["shots"],"prompt":row["prompt"],
               "raw":raw,"expected":row["expected"],**score_output(row["task"],raw,row["expected"])})
            save_json("gpt3_raw.json",gpt3_rows)
        display(pd.DataFrame(gpt3_rows).groupby(["task","shots"])[["content_correct","strict_match"]].mean())
        combined=pd.concat([local_table[["model","task","shots","content_correct","strict_match"]],
                            pd.DataFrame(gpt3_rows)[["model","task","shots","content_correct","strict_match"]]],ignore_index=True)
        all_scores=combined.groupby(["model","task","shots"])[["content_correct","strict_match"]].mean()
        display(all_scores);all_scores.to_csv(RESULT_DIR/"all_models_comparison.csv")
        save_json("gpt3_status.json",{"status":"completed","model":GPT3_MODEL_ID,"evidence":GPT3_MODEL_EVIDENCE,"calls":len(gpt3_rows)})
        PASSED.add("4-4_gpt3_direct")
    except Exception:
        save_json("gpt3_status.json",{"status":"incomplete","completed_calls":len(gpt3_rows)})
        raise
    finally:
        del api_key
else:
    save_json("gpt3_status.json",{"status":"not_run","reason":"원형 GPT-3 접근 경로 미설정"})
    print("GPT-3 원형: 미실행 · 다른 모델 결과로 대체하지 않음")


### 제출 답안

- 아래 문자열에 직접 기록 · 표/그림은 위 실행 결과와 함께 제출
- 관찰과 해석을 구분하고, 예상과 다른 결과도 그대로 기록


In [ ]:
ANSWERS["4-4"] = {
    "원형 확인 근거 또는 미실행 사유": "",
    "동일 조건 비교 결과 또는 실행 계획": "",
    "few-shot 시 가중치 변경 여부": "",
    "관찰 범위와 일반화의 한계": ""
}
save_json("answers.json", ANSWERS)


## 제출 — 마지막 10분

- 학번·이름과 문제별 답안 확인
- 노트북: Colab 파일 → 다운로드 → .ipynb 다운로드
- 결과 ZIP: 아래 셀 실행 후 다운로드
- 답안 빈칸과 미실행 항목은 manifest에 기록됨
- 수치 검사 통과가 서술 답안의 정답 판정을 의미하지 않음


In [ ]:
EXPECTED_QUESTIONS=['4-1', '4-2', '4-3', '4-4']

missing=[f"{key}/{field}" for key,entry in ANSWERS.items() for field,value in entry.items() if not str(value).strip()]
missing += [key+"/답안 셀 미실행" for key in EXPECTED_QUESTIONS if key not in ANSWERS]
manifest={"student_id":STUDENT_ID,"student_name":STUDENT_NAME,"environment":ENV,
          "checks_passed":sorted(PASSED),"unanswered":missing,
          "identity_complete":bool(STUDENT_ID.strip() and STUDENT_NAME.strip())}
save_json("answers.json",ANSWERS); save_json("manifest.json",manifest)
print("미작성 답안:",missing)
print("학번/이름 입력:",manifest["identity_complete"])


In [ ]:
download_results("week4_submission")
